## CPSC 4970 Homework 5 - Jonathan Braun

In [1]:
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

pd.set_option("display.max_colwidth", 120)

## Load and label the data

In [2]:
def load_headlines(path):
    path = Path(path)
    with path.open("r", encoding="utf-8", errors="ignore") as file:
        return [line.strip() for line in file if line.strip()]

clickbait = load_headlines("clickbait_data")
non_clickbait = load_headlines("non_clickbait_data")

print(f"Clickbait headlines: {len(clickbait):,}")
print(f"Non-clickbait headlines: {len(non_clickbait):,}")
print(f"Total headlines: {len(clickbait) + len(non_clickbait):,}")

Clickbait headlines: 15,999
Non-clickbait headlines: 16,001
Total headlines: 32,000


## Create feature list and target vector

In [3]:
X = clickbait + non_clickbait
y = np.array([1] * len(clickbait) + [0] * len(non_clickbait))

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training examples: {len(X_train):,}")
print(f"Testing examples: {len(X_test):,}")
print(f"Training class balance: {np.bincount(y_train)}")
print(f"Testing class balance: {np.bincount(y_test)}")

Training examples: 25,600
Testing examples: 6,400
Training class balance: [12801 12799]
Testing class balance: [3200 3200]


## Feature representation and cross-validation setup

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1"
}

tfidf = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2)
)

## Model 1: k-nearest neighbors

In [ ]:
knn_pipeline = Pipeline([
    ("tfidf", tfidf),
    ("classifier", KNeighborsClassifier())
])

knn_params = {
    "classifier__n_neighbors": [3, 5, 7, 11, 15, 21, 31],
    "classifier__weights": ["uniform", "distance"]
}

knn_grid = GridSearchCV(
    knn_pipeline,
    knn_params,
    cv=cv,
    scoring=scoring,
    refit="f1",
    n_jobs=-1,
    return_train_score=False
)

knn_grid.fit(X_train, y_train)

print("Best kNN parameters:")
print(knn_grid.best_params_)
print(f"Best kNN CV F1 score: {knn_grid.best_score_:.4f}")